In [42]:
  !git clone https://github.com/mkobycheva/recommendation-system.git 2>/dev/null \
  || (cd recommendation-system && git pull)

%cd recommendation-system
!git checkout transformer && git pull
!pip install -r requirements.txt -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

/content/recommendation-system/recommendation-system/recommendation-system/recommendation-system
Branch 'transformer' set up to track remote branch 'transformer' from 'origin'.
Switched to a new branch 'transformer'
Already up to date.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
import os
import numpy as np
import pandas as pd

from src.evaluation.metrics import ndcg_at_k, recall_at_k, map_at_k

# Data preparation

In [44]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [45]:
from src.data.build_matrix import add_domain_item_ids
from sklearn.preprocessing import LabelEncoder

books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test  = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test  = add_domain_item_ids(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df  = pd.concat([books_test, movies_test], ignore_index=True)

# 5-core item filtering
# Only train_df decides which items "count" — valid/test must never leak info
# about which items are frequent, they can only inherit train's decision.
MIN_ITEM_FREQ = 5
item_counts_train = train_df['item_id'].value_counts()
frequent_items = set(item_counts_train[item_counts_train >= MIN_ITEM_FREQ].index)

print(f"Items before filtering: {train_df['item_id'].nunique()}")
print(f"Items after filtering (freq >= {MIN_ITEM_FREQ}): {len(frequent_items)}")

rows_before = {name: len(df) for name, df in [('train', train_df), ('valid', valid_df), ('test', test_df)]}

train_df = train_df[train_df['item_id'].isin(frequent_items)].reset_index(drop=True)
valid_df = valid_df[valid_df['item_id'].isin(frequent_items)].reset_index(drop=True)
test_df  = test_df[test_df['item_id'].isin(frequent_items)].reset_index(drop=True)

for name, df, before in [('train', train_df, rows_before['train']),
                          ('valid', valid_df, rows_before['valid']),
                          ('test', test_df, rows_before['test'])]:
    print(f"{name}: {before} -> {len(df)} rows ({100*len(df)/before:.1f}% kept)")

# --- Rebuild vocab on the FILTERED data only, so NUM_ITEMS actually shrinks ---
encoder = LabelEncoder()
encoder.fit(pd.concat([train_df['item_id'], valid_df['item_id'], test_df['item_id']]).unique())

NUM_ITEMS = len(encoder.classes_) + 1
print(f"NUM_ITEMS = {NUM_ITEMS}")

for df in (train_df, valid_df, test_df):
    df['item_idx'] = encoder.transform(df['item_id']) + 1

Items before filtering: 587064
Items after filtering (freq >= 5): 190466
train: 3750813 -> 2894431 rows (77.2% kept)
valid: 254376 -> 181182 rows (71.2% kept)
test: 254376 -> 172979 rows (68.0% kept)
NUM_ITEMS = 190467


In [46]:
MAX_LEN = 50

def build_user_sequences(df):
    """user_id -> list of item_idx, sorted by timestamp"""
    df_sorted = df.sort_values(['user_id', 'timestamp'])
    return df_sorted.groupby('user_id')['item_idx'].apply(list).to_dict()

def generate_sliding_windows(user_sequences, max_len=MAX_LEN, min_len=2):
    """
    Next-item prediction на кожній позиції одразу (SASRec-style).
    input:  [0,...,0, i1,i2,i3,i4]
    target: [0,...,0, i2,i3,i4,i5]
    """
    X, Y = [], []
    for seq in user_sequences.values():
        if len(seq) < min_len:
            continue
        inp = seq[:-1]
        tgt = seq[1:]
        if len(inp) >= max_len:
            inp = inp[-max_len:]
            tgt = tgt[-max_len:]
        else:
            pad = max_len - len(inp)
            inp = [0] * pad + inp
            tgt = [0] * pad + tgt
        X.append(inp)
        Y.append(tgt)
    return np.array(X), np.array(Y)

def build_val_eval_batch(user_sequences, max_len=MAX_LEN):
    """Isolates only the true held-out validation target per user,
    mirroring build_test_eval_batch — avoids contamination from
    train-only transitions inside the combined sequence."""
    X, Y = [], []
    for seq in user_sequences.values():
        if len(seq) < 2:
            continue
        inp = seq[:-1]
        target = seq[-1]
        if len(inp) >= max_len:
            inp = inp[-max_len:]
        else:
            pad = max_len - len(inp)
            inp = [0] * pad + inp
        X.append(inp)
        Y.append(target)
    return np.array(X), np.array(Y)

train_sequences = build_user_sequences(train_df)
X_train, Y_train = generate_sliding_windows(train_sequences)

valid_combined = pd.concat([train_df, valid_df])
valid_sequences = build_user_sequences(valid_combined)
X_val, y_val = build_val_eval_batch(valid_sequences, max_len=MAX_LEN)  # renamed Y_val -> y_val (single target per user, not per-position)

print(f"Train: X={X_train.shape}, Y={Y_train.shape}")
print(f"Valid: X={X_val.shape}, y={len(y_val)}")

Train: X=(127132, 50), Y=(127132, 50)
Valid: X=(127170, 50), y=127170


In [47]:
non_pad_train = (Y_train != 0).sum()
total_train = Y_train.size
print(f"Non-padding targets: {non_pad_train} out of {total_train} ({100*non_pad_train/total_train:.1f}%)")

Non-padding targets: 2204414 out of 6356600 (34.7%)


# SASRec

In [48]:
import torch
from torch.utils.data import Dataset, DataLoader

class SASRecDataset(Dataset):
    """Wraps padded (input, target) sequences for next-item prediction."""

    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.Y = torch.tensor(Y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


BATCH_SIZE = 128
train_loader = DataLoader(SASRecDataset(X_train, Y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_val, dtype=torch.long), torch.tensor(y_val, dtype=torch.long)),
    batch_size=BATCH_SIZE, shuffle=False
)  # y_val is one target per user now, not a full sequence -> use TensorDataset, same as test_loader

for x_batch, y_batch in train_loader:
    print(f"X batch shape: {x_batch.shape}, Y batch shape: {y_batch.shape}")
    break

X batch shape: torch.Size([128, 50]), Y batch shape: torch.Size([128, 50])


In [49]:
# Build a train-eval loader mirroring build_val_eval_batch: only the LAST position
# per user, so train MAP@5 is measured the same way as val MAP@5 (apples-to-apples).
# Using the full per-position X_train/Y_train for MAP@5 would be misleadingly easy,
# since many of those positions are the same transitions the model was trained on.
X_train_eval, y_train_eval = build_val_eval_batch(train_sequences, max_len=MAX_LEN)

train_eval_loader = DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_train_eval, dtype=torch.long), torch.tensor(y_train_eval, dtype=torch.long)),
    batch_size=BATCH_SIZE, shuffle=False
)

print(f"Train-eval: X={X_train_eval.shape}, y={len(y_train_eval)}")

Train-eval: X=(127132, 50), y=127132


In [50]:
import torch.nn as nn

class SASRec(nn.Module):
    def __init__(self, num_items, max_len=MAX_LEN, d_model=64, n_heads=2, n_layers=2, dropout=0.2, emb_dropout=0.5):
        super().__init__()
        self.item_emb = nn.Embedding(num_items, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.emb_dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.layer_norm = nn.LayerNorm(d_model)
        self.max_len = max_len

    def forward(self, item_seq):
        batch_size, seq_len = item_seq.shape
        positions = torch.arange(seq_len, device=item_seq.device).unsqueeze(0).expand(batch_size, -1)

        x = self.item_emb(item_seq) + self.pos_emb(positions)
        x = self.emb_dropout(x)

        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(item_seq.device)
        padding_mask = (item_seq == 0)

        x = self.encoder(x, mask=causal_mask, src_key_padding_mask=padding_mask)

        # padding-only query positions can produce NaN internally (fully-masked attention row);
        # those positions are excluded from the loss anyway, so zeroing NaNs here is safe
        # and stops them from contaminating real positions in later computations
        x = torch.nan_to_num(x, nan=0.0)

        x = self.layer_norm(x)
        return x

    def predict_logits(self, hidden_states):
        return hidden_states @ self.item_emb.weight.T

In [51]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SASRec(num_items=NUM_ITEMS, max_len=MAX_LEN, d_model=64, n_heads=2, n_layers=2, dropout=0.2).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=1, factor=0.5)

print(f"Device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
item_domain = np.array(['pad'] + [
    'books' if c.startswith('books::') else 'movies'
    for c in encoder.classes_
])

domain_id = torch.tensor(
    [0 if d == 'pad' else (1 if d == 'books' else 2) for d in item_domain],
    device=device
)
books_item_idx = torch.tensor(np.where(item_domain == 'books')[0], device=device)
movies_item_idx = torch.tensor(np.where(item_domain == 'movies')[0], device=device)

print(f"Books items: {len(books_item_idx)}, Movies items: {len(movies_item_idx)}")

In [ ]:
# # Popularity-weighted negative sampling: negatives should be sampled proportional
# # to how often each item appears in train, not uniformly. Uniform negatives are
# # almost always obscure long-tail items — trivially easy to distinguish from the
# # true target — so the model never learns to out-rank genuinely popular competitors,
# # which is exactly why it currently fails to beat the popularity baseline.
# item_pop_counts = train_df['item_idx'].value_counts()
# full_pop_counts = np.zeros(NUM_ITEMS)
# full_pop_counts[item_pop_counts.index.values] = item_pop_counts.values
# full_pop_counts_t = torch.tensor(full_pop_counts, device=device, dtype=torch.float)

# # smoothing exponent (0.75, as in word2vec negative sampling) prevents the
# # most popular items from dominating the negative pool entirely
# POP_SMOOTHING = 0.75
# smoothed_pop = full_pop_counts_t.clamp(min=1.0) ** POP_SMOOTHING

# books_neg_weights = smoothed_pop[books_item_idx]
# movies_neg_weights = smoothed_pop[movies_item_idx]

# print(f"Books weight range: {books_neg_weights.min().item():.1f} - {books_neg_weights.max().item():.1f}")
# print(f"Movies weight range: {movies_neg_weights.min().item():.1f} - {movies_neg_weights.max().item():.1f}")

In [ ]:
def sampled_logits_and_labels(model, hidden_valid, targets_valid, num_items, num_negatives=100):
    N = hidden_valid.size(0)
    device = hidden_valid.device

    # Sample negatives from the SAME domain as the target, so the model can't
    # shortcut on "is this a book or a movie" instead of learning real relevance
    target_domain_id = domain_id[targets_valid]
    is_books = target_domain_id == 1

    books_sample_idx = torch.randint(0, len(books_item_idx), (N, num_negatives), device=device)
    movies_sample_idx = torch.randint(0, len(movies_item_idx), (N, num_negatives), device=device)
    neg_items = torch.where(
        is_books.unsqueeze(1),
        books_item_idx[books_sample_idx],
        movies_item_idx[movies_sample_idx],
    )

    collision = neg_items == targets_valid.unsqueeze(1)
    if collision.any():
        n_coll = int(collision.sum().item())
        fallback_books = books_item_idx[torch.randint(0, len(books_item_idx), (n_coll,), device=device)]
        fallback_movies = movies_item_idx[torch.randint(0, len(movies_item_idx), (n_coll,), device=device)]
        is_books_flat = is_books.unsqueeze(1).expand(-1, num_negatives)[collision]
        neg_items[collision] = torch.where(is_books_flat, fallback_books, fallback_movies)

    candidates = torch.cat([targets_valid.unsqueeze(1), neg_items], dim=1)   # (N, 1+K)
    cand_emb = model.item_emb(candidates)                                    # (N, 1+K, d_model)

    logits = torch.bmm(cand_emb, hidden_valid.unsqueeze(-1)).squeeze(-1)     # (N, 1+K)
    labels = torch.zeros(N, dtype=torch.long, device=device)                 # correct item is always at index 0

    return logits, labels

In [ ]:
@torch.no_grad()
def evaluate_map_at_k(model, hidden_valid, targets_valid, num_items, num_negatives=100, k=5):
    """
    Builds candidate-pool rankings and scores them with the repo's map_at_k
    (relies on average_precision_at_k under the hood).
    Each "user" here is really one (position, target) example — with a single
    relevant item — matching the interface map_at_k expects.
    """
    model.eval()
    N = hidden_valid.size(0)
    device = hidden_valid.device

    neg_items = torch.randint(1, num_items, (N, num_negatives), device=device)
    collision = neg_items == targets_valid.unsqueeze(1)
    if collision.any():
        neg_items[collision] = torch.randint(1, num_items, (int(collision.sum().item()),), device=device)

    candidates = torch.cat([targets_valid.unsqueeze(1), neg_items], dim=1)   # (N, 1+K)
    cand_emb = model.item_emb(candidates)
    logits = torch.bmm(cand_emb, hidden_valid.unsqueeze(-1)).squeeze(-1)      # (N, 1+K)

    top_k_idx = logits.topk(k, dim=1).indices                                # (N, k), indices into candidate axis
    top_k_items = torch.gather(candidates, 1, top_k_idx)                     # (N, k), actual item ids

    recommended_items_by_user = {i: top_k_items[i].tolist() for i in range(N)}
    relevant_items_by_user = {i: [targets_valid[i].item()] for i in range(N)}

    return map_at_k(recommended_items_by_user, relevant_items_by_user, k=k)

In [ ]:
@torch.no_grad()
def sampled_ranking_map_at_k(model, hidden_all, targets_all, num_items, num_negatives=999, k=5, seed=42, chunk_size=2048):
    model.eval()
    device = hidden_all.device
    generator = torch.Generator(device=device).manual_seed(seed)

    recommended_items_by_user = {}
    relevant_items_by_user = {}
    global_idx = 0

    for start in range(0, hidden_all.size(0), chunk_size):
        end = start + chunk_size
        hidden_chunk = hidden_all[start:end]
        targets_chunk = targets_all[start:end]
        n = hidden_chunk.size(0)

        target_domain_id_chunk = domain_id[targets_chunk]
        is_books_chunk = target_domain_id_chunk == 1
        books_sample_idx = torch.randint(0, len(books_item_idx), (n, num_negatives), device=device, generator=generator)
        movies_sample_idx = torch.randint(0, len(movies_item_idx), (n, num_negatives), device=device, generator=generator)
        neg_items = torch.where(
            is_books_chunk.unsqueeze(1),
            books_item_idx[books_sample_idx],
            movies_item_idx[movies_sample_idx],
        )

        collision = neg_items == targets_chunk.unsqueeze(1)
        if collision.any():
            n_coll = int(collision.sum().item())
            fallback_books = books_item_idx[torch.randint(0, len(books_item_idx), (n_coll,), device=device, generator=generator)]
            fallback_movies = movies_item_idx[torch.randint(0, len(movies_item_idx), (n_coll,), device=device, generator=generator)]
            is_books_flat = is_books_chunk.unsqueeze(1).expand(-1, num_negatives)[collision]
            neg_items[collision] = torch.where(is_books_flat, fallback_books, fallback_movies)

        candidates = torch.cat([targets_chunk.unsqueeze(1), neg_items], dim=1)   # (n, 1+K)
        cand_emb = model.item_emb(candidates)
        logits = torch.bmm(cand_emb, hidden_chunk.unsqueeze(-1)).squeeze(-1)      # (n, 1+K)

        top_k_idx = logits.topk(k, dim=1).indices
        top_k_items = torch.gather(candidates, 1, top_k_idx)

        for i in range(n):
            recommended_items_by_user[global_idx] = top_k_items[i].tolist()
            relevant_items_by_user[global_idx] = [targets_chunk[i].item()]
            global_idx += 1

    return map_at_k(recommended_items_by_user, relevant_items_by_user, k=k)

## Training

In [ ]:
import wandb

wandb.init(
    project="recsys-sequential",
    name="sasrec-v4-dmodel64",
    config={
        "model": "SASRec",
        "max_len": MAX_LEN,
        "d_model": 64,
        "n_heads": 2,
        "n_layers": 2,
        "dropout": 0.2,
        "batch_size": BATCH_SIZE,
        "lr": 1e-3,
        "weight_decay": 1e-4,
    }
)

In [ ]:
EPOCHS = 30
PATIENCE = 5
MIN_DELTA = 0.0
MIN_EPOCHS_BEFORE_STOPPING = 10
WARMUP_EPOCHS = 5
best_val_map = -float('inf')
patience_counter = 0
best_state = None

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_grad_norm = 0.0

    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        hidden = model(x_batch)

        # Flatten and keep only non-padding positions BEFORE the expensive matmul
        hidden_flat = hidden.view(-1, hidden.size(-1))
        targets_flat = y_batch.view(-1)
        mask = targets_flat != 0
        hidden_valid = hidden_flat[mask]
        targets_valid = targets_flat[mask]

        logits, labels = sampled_logits_and_labels(model, hidden_valid, targets_valid, NUM_ITEMS, num_negatives=100)
        loss = criterion(logits, labels)
        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        total_grad_norm += grad_norm.item()

        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    avg_grad_norm = total_grad_norm / len(train_loader)

    model.eval()
    val_loss = 0.0
    all_hidden = []
    all_targets = []

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            hidden = model(x_batch)

            # y_batch is now a single held-out target per user -> take only the LAST position's hidden state,
            # matching how build_val_eval_batch left-pads (last position = last real history item -> predicts y_batch)
            hidden_valid = hidden[:, -1, :]
            targets_valid = y_batch

            all_hidden.append(hidden_valid.cpu())
            all_targets.append(targets_valid.cpu())

            logits, labels = sampled_logits_and_labels(model, hidden_valid, targets_valid, NUM_ITEMS, num_negatives=100)
            loss = criterion(logits, labels)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    all_hidden = torch.cat(all_hidden).to(device)
    all_targets = torch.cat(all_targets).to(device)
    val_map5 = sampled_ranking_map_at_k(model, all_hidden, all_targets, NUM_ITEMS, num_negatives=999, k=5, seed=42)

    # Train MAP@5 for overfit tracking — same last-position-only protocol as val,
    # so the train/val gap is a real generalization gap, not a measurement artifact
    all_hidden_train_eval, all_targets_train_eval = [], []
    with torch.no_grad():
        for x_batch, y_batch in train_eval_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            hidden = model(x_batch)
            all_hidden_train_eval.append(hidden[:, -1, :].cpu())
            all_targets_train_eval.append(y_batch.cpu())

    all_hidden_train_eval = torch.cat(all_hidden_train_eval).to(device)
    all_targets_train_eval = torch.cat(all_targets_train_eval).to(device)
    train_map5 = sampled_ranking_map_at_k(model, all_hidden_train_eval, all_targets_train_eval, NUM_ITEMS, num_negatives=999, k=5, seed=42)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "grad_norm": avg_grad_norm,
        "learning_rate": current_lr,
        "val_map@5": val_map5,
        "train_map@5": train_map5,
        "overfit_gap_map@5": train_map5 - val_map5,
    })

    print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train MAP@5: {train_map5:.4f} | Val MAP@5: {val_map5:.4f} | Gap: {train_map5 - val_map5:.4f} | Grad Norm: {avg_grad_norm:.4f}")

    if epoch + 1 > WARMUP_EPOCHS and val_map5 > best_val_map + MIN_DELTA:
        best_val_map = val_map5
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if epoch + 1 >= MIN_EPOCHS_BEFORE_STOPPING and patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(best_state)
wandb.finish()

## Validate on test set

In [ ]:
test_combined = pd.concat([train_df, valid_df, test_df])
test_sequences = build_user_sequences(test_combined)

def build_test_eval_batch(user_sequences, max_len=MAX_LEN):
    """
    For each user: history = everything except the last interaction,
    target = the last interaction (the held-out test item to predict).
    Left-padding guarantees the last position in each row is always
    the last real history item (never padding), as long as history has >=1 item.
    """
    X, Y = [], []
    for seq in user_sequences.values():
        if len(seq) < 2:
            continue
        inp = seq[:-1]
        target = seq[-1]
        if len(inp) >= max_len:
            inp = inp[-max_len:]
        else:
            pad = max_len - len(inp)
            inp = [0] * pad + inp
        X.append(inp)
        Y.append(target)
    return np.array(X), np.array(Y)

X_test, y_test = build_test_eval_batch(test_sequences)
print(f"Test: X={X_test.shape}, y={len(y_test)}")

In [ ]:
model.eval()

test_loader = DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_test, dtype=torch.long), torch.tensor(y_test, dtype=torch.long)),
    batch_size=BATCH_SIZE, shuffle=False
)

all_hidden_test, all_targets_test = [], []
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        hidden = model(x_batch)
        all_hidden_test.append(hidden[:, -1, :].cpu())   # last position = prediction for the held-out item
        all_targets_test.append(y_batch)

all_hidden_test = torch.cat(all_hidden_test).to(device)
all_targets_test = torch.cat(all_targets_test).to(device)

print(f"all_hidden_test: {all_hidden_test.shape}, all_targets_test: {all_targets_test.shape}")

In [ ]:
def average_metric_over_users(metric_fn, recommended_by_user, relevant_by_user, k):
    """Applies a per-user metric function (ndcg_at_k, recall_at_k, precision_at_k)
    across a dict of users and averages the result — mirrors what map_at_k does
    internally via average_precision_at_k, since those other functions don't have
    their own dict-aware wrapper in the repo.
    """
    scores = [
        metric_fn(recommended_by_user.get(user_id, []), relevant_items, k=k)
        for user_id, relevant_items in relevant_by_user.items()
    ]
    return sum(scores) / len(scores) if scores else 0.0

In [ ]:
import gc

if 'optimizer' in globals():
    del optimizer
if 'scheduler' in globals():
    del scheduler

gc.collect()
torch.cuda.empty_cache()

print(f"Free GPU memory: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

@torch.no_grad()
def full_ranking_eval(model, hidden_all, targets_all, num_items, k_list=(5, 10), chunk_size=2048):
    """Returns top-k recommended items dict + relevant items dict, ready for map_at_k/ndcg_at_k/recall_at_k."""
    max_k = max(k_list)
    all_item_emb = model.item_emb.weight
    recommended_items_by_user = {}
    relevant_items_by_user = {}
    global_idx = 0

    for start in range(0, hidden_all.size(0), chunk_size):
        end = start + chunk_size
        hidden_chunk = hidden_all[start:end]
        targets_chunk = targets_all[start:end]

        logits_chunk = hidden_chunk @ all_item_emb.T

        # Restrict candidates to the same domain as each row's target item
        # (also implicitly excludes padding, since domain_id[0] == 0 and no real target has domain_id 0)
        target_domain_id_chunk = domain_id[targets_chunk]                            # (n,)
        wrong_domain_mask = domain_id.unsqueeze(0) != target_domain_id_chunk.unsqueeze(1)  # (n, num_items)
        logits_chunk = logits_chunk.masked_fill(wrong_domain_mask, float('-inf'))

        top_k_items = logits_chunk.topk(max_k, dim=1).indices

        for i in range(hidden_chunk.size(0)):
            recommended_items_by_user[global_idx] = top_k_items[i].tolist()
            relevant_items_by_user[global_idx] = [targets_chunk[i].item()]
            global_idx += 1

    return recommended_items_by_user, relevant_items_by_user

recommended_model, relevant_items = full_ranking_eval(model, all_hidden_test, all_targets_test, NUM_ITEMS,  chunk_size=128)

model_metrics = {
    "MAP@5": map_at_k(recommended_model, relevant_items, k=5),
    "NDCG@10": average_metric_over_users(ndcg_at_k, recommended_model, relevant_items, k=10),
    "Recall@10": average_metric_over_users(recall_at_k, recommended_model, relevant_items, k=10),
}
print("Model:", model_metrics)

In [ ]:
books_item_idx_np = books_item_idx.cpu().numpy()
movies_item_idx_np = movies_item_idx.cpu().numpy()

top_popular_books = train_df.loc[train_df['item_idx'].isin(books_item_idx_np), 'item_idx'].value_counts().index[:10].tolist()
top_popular_movies = train_df.loc[train_df['item_idx'].isin(movies_item_idx_np), 'item_idx'].value_counts().index[:10].tolist()

rng = np.random.default_rng(42)
recommended_popular = {}
recommended_random = {}
for i, target_item in enumerate(y_test):
    is_books_target = item_domain[target_item] == 'books'
    recommended_popular[i] = top_popular_books if is_books_target else top_popular_movies
    pool = books_item_idx_np if is_books_target else movies_item_idx_np
    recommended_random[i] = rng.choice(pool, size=10, replace=False).tolist()

# assumes model_metrics, popular_metrics, random_metrics dicts already computed, e.g.:
# model_metrics = {"MAP@5": ..., "NDCG@10": ..., "Recall@10": ...}
popular_metrics = {
    "MAP@5": map_at_k(recommended_popular, relevant_items, k=5),
    "NDCG@10": average_metric_over_users(ndcg_at_k, recommended_popular, relevant_items, k=10),
    "Recall@10": average_metric_over_users(recall_at_k, recommended_popular, relevant_items, k=10),
}
random_metrics = {
    "MAP@5": map_at_k(recommended_random, relevant_items, k=5),
    "NDCG@10": average_metric_over_users(ndcg_at_k, recommended_random, relevant_items, k=10),
    "Recall@10": average_metric_over_users(recall_at_k, recommended_random, relevant_items, k=10),
}

rows = []
for metric in model_metrics:
    model_val = model_metrics[metric]
    popular_val = popular_metrics[metric]
    random_val = random_metrics[metric]

    lift_popular = model_val / popular_val if popular_val > 0 else float('inf')
    lift_random = model_val / random_val if random_val > 0 else float('inf')

    rows.append({
        "Metric": metric,
        "Model": model_val,
        "Top-Popular": popular_val,
        "Random": random_val,
        "Lift vs Popular": lift_popular,
        "Lift vs Random": lift_random,
    })

df = pd.DataFrame(rows)

# nice formatting: small decimals for raw metrics, "x" suffix for lift, inf handled gracefully
formatted = df.copy()
for col in ["Model", "Top-Popular", "Random"]:
    formatted[col] = formatted[col].map(lambda v: f"{v:.6f}")
for col in ["Lift vs Popular", "Lift vs Random"]:
    formatted[col] = formatted[col].map(lambda v: "∞" if v == float('inf') else f"×{v:.2f}")

print(formatted.to_string(index=False))

In [ ]:
def split_by_domain(relevant_items_by_user, item_domain):
    """Splits user indices into 'books' and 'movies' groups based on
    the domain of their relevant (held-out target) item."""
    books_users, movies_users = [], []
    for user_id, relevant in relevant_items_by_user.items():
        target_item = relevant[0]
        if item_domain[target_item] == 'books':
            books_users.append(user_id)
        else:
            movies_users.append(user_id)
    return books_users, movies_users


def subset_dict(d, user_ids):
    return {u: d[u] for u in user_ids}


def compute_metrics_for_subset(recommended, relevant, user_ids):
    rec_subset = subset_dict(recommended, user_ids)
    rel_subset = subset_dict(relevant, user_ids)
    return {
        "MAP@5": map_at_k(rec_subset, rel_subset, k=5),
        "NDCG@10": average_metric_over_users(ndcg_at_k, rec_subset, rel_subset, k=10),
        "Recall@10": average_metric_over_users(recall_at_k, rec_subset, rel_subset, k=10),
    }


books_users, movies_users = split_by_domain(relevant_items, item_domain)
print(f"Books users: {len(books_users)}, Movies users: {len(movies_users)}")

results_by_domain = {}
for domain_name, user_ids in (("Books", books_users), ("Movies", movies_users)):
    model_d = compute_metrics_for_subset(recommended_model, relevant_items, user_ids)
    popular_d = compute_metrics_for_subset(recommended_popular, relevant_items, user_ids)
    random_d = compute_metrics_for_subset(recommended_random, relevant_items, user_ids)

    rows = []
    for metric in model_d:
        model_val = model_d[metric]
        popular_val = popular_d[metric]
        random_val = random_d[metric]
        lift_popular = model_val / popular_val if popular_val > 0 else float('inf')
        lift_random = model_val / random_val if random_val > 0 else float('inf')
        rows.append({
            "Metric": metric,
            "Model": model_val,
            "Top-Popular": popular_val,
            "Random": random_val,
            "Lift vs Popular": lift_popular,
            "Lift vs Random": lift_random,
        })

    df_domain = pd.DataFrame(rows)
    formatted = df_domain.copy()
    for col in ["Model", "Top-Popular", "Random"]:
        formatted[col] = formatted[col].map(lambda v: f"{v:.6f}")
    for col in ["Lift vs Popular", "Lift vs Random"]:
        formatted[col] = formatted[col].map(lambda v: "∞" if v == float('inf') else f"×{v:.2f}")

    print(f"\n=== {domain_name} ===")
    print(formatted.to_string(index=False))
    results_by_domain[domain_name] = df_domain

## Saving metrics results

In [ ]:
def extract_metric(df_domain, column, metric):
    return float(df_domain.set_index('Metric')[column][metric])

def sasrec_result_row(model_name, column):
    row = {
        'model': model_name,
        'd_model': 64, 'n_heads': 2, 'n_layers': 2,
        'k': 10, 'map_k': 5,
        'n_users': train_df['user_id'].nunique(),
        'n_items': NUM_ITEMS - 1,
    }
    for metric_label, key in [('MAP@5', 'map@5'), ('NDCG@10', 'ndcg@10'), ('Recall@10', 'recall@10')]:
        books_val = extract_metric(results_by_domain['Books'], column, metric_label)
        movies_val = extract_metric(results_by_domain['Movies'], column, metric_label)
        row[f'books_test_{key}'] = round(books_val, 4)
        row[f'movies_test_{key}'] = round(movies_val, 4)
        row[f'test_macro_{key}'] = round((books_val + movies_val) / 2, 4)
    return row

sasrec_metrics_results = pd.DataFrame([
    sasrec_result_row('sasrec_books_movies_joint', 'Model'),
    sasrec_result_row('popularity_baseline', 'Top-Popular'),
    sasrec_result_row('random_baseline', 'Random'),
])

RESULTS_DIR = f'{DATA_DIR}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
sasrec_metrics_results.to_csv(f'{RESULTS_DIR}/sasrec_metrics.csv', index=False)
sasrec_metrics_results

## Export artifacts for the app MVP


In [ ]:
import json

ARTIFACTS_DIR = f'{DATA_DIR}/artifacts/sasrec'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

torch.save(model.state_dict(), f'{ARTIFACTS_DIR}/model_state_dict.pt')

item2idx = {cls: i + 1 for i, cls in enumerate(encoder.classes_)}
with open(f'{ARTIFACTS_DIR}/item2idx.json', 'w') as f:
    json.dump(item2idx, f)

with open(f'{ARTIFACTS_DIR}/domain_item_indices.json', 'w') as f:
    json.dump({
        'books': books_item_idx.cpu().tolist(),
        'movies': movies_item_idx.cpu().tolist(),
    }, f)

with open(f'{ARTIFACTS_DIR}/config.json', 'w') as f:
    json.dump({
        'num_items': NUM_ITEMS,
        'max_len': MAX_LEN,
        'd_model': 64,
        'n_heads': 2,
        'n_layers': 2,
        'dropout': 0.2,
    }, f)

print(f'Saved SASRec artifacts to {ARTIFACTS_DIR}')


# Bert4Rec

In [ ]:
# --- BERT4Rec: masked (Cloze) training data + leave-one-out eval data ---

MASK_TOKEN = NUM_ITEMS  # new id beyond real items (0=padding, 1..NUM_ITEMS-1=items, NUM_ITEMS=[MASK])
MASK_PROB = 0.2         # fraction of REAL (non-padding) positions masked per sequence

def build_bert4rec_training_batch(user_sequences, max_len=MAX_LEN, mask_prob=MASK_PROB, rng=None):
    """
    Cloze-style masking (as in original BERT4Rec paper):
      - pick ~mask_prob of real (non-padding) positions per sequence
      - of those: 80% -> replace with [MASK], 10% -> replace with random item, 10% -> leave unchanged
      - target at masked positions = original item id; target=0 elsewhere (ignored in loss, same
        convention as padding_idx=0, since no real item has index 0)
    Regenerate this EVERY epoch (dynamic masking) — re-masking the same sequences differently
    each epoch is a meaningful data augmentation and is standard practice for BERT4Rec.
    """
    rng = rng or np.random.default_rng()
    X, Y = [], []
    for seq in user_sequences.values():
        if len(seq) < 2:
            continue
        seq = seq[-max_len:] if len(seq) > max_len else seq
        pad = max_len - len(seq)
        inp = [0] * pad + list(seq)
        tgt = [0] * max_len

        real_positions = list(range(pad, max_len))
        n_mask = max(1, int(round(len(real_positions) * mask_prob)))
        mask_positions = rng.choice(real_positions, size=min(n_mask, len(real_positions)), replace=False)

        for i in mask_positions:
            tgt[i] = inp[i]
            r = rng.random()
            if r < 0.8:
                inp[i] = MASK_TOKEN
            elif r < 0.9:
                inp[i] = int(rng.integers(1, NUM_ITEMS))  # random real item, never MASK/padding
            # else: leave unchanged (model still must predict tgt[i] correctly)

        X.append(inp)
        Y.append(tgt)
    return np.array(X), np.array(Y)


def build_bert4rec_eval_batch(user_sequences, max_len=MAX_LEN, mask_token=MASK_TOKEN):
    """
    Leave-one-out eval, BERT4Rec style: history (everything except the held-out item)
    gets a [MASK] appended at the END, left-padded. The hidden state at the LAST position
    (the [MASK] position) is used for prediction — this keeps hidden[:, -1, :] downstream
    code identical to what SASRec eval already uses.
    """
    X, Y = [], []
    for seq in user_sequences.values():
        if len(seq) < 2:
            continue
        inp_hist = seq[:-1]
        target = seq[-1]
        if len(inp_hist) >= max_len - 1:       # leave room for the appended [MASK]
            inp_hist = inp_hist[-(max_len - 1):]
        seq_with_mask = list(inp_hist) + [mask_token]
        pad = max_len - len(seq_with_mask)
        X.append([0] * pad + seq_with_mask)
        Y.append(target)
    return np.array(X), np.array(Y)


# Build eval sets (static — no masking randomness needed here, same as SASRec's build_val_eval_batch)
X_val_bert, y_val_bert = build_bert4rec_eval_batch(valid_sequences, max_len=MAX_LEN)
X_train_eval_bert, y_train_eval_bert = build_bert4rec_eval_batch(train_sequences, max_len=MAX_LEN)

print(f"Val (BERT4Rec eval): X={X_val_bert.shape}, y={len(y_val_bert)}")
print(f"Train-eval (BERT4Rec eval): X={X_train_eval_bert.shape}, y={len(y_train_eval_bert)}")

val_loader_bert = DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_val_bert, dtype=torch.long), torch.tensor(y_val_bert, dtype=torch.long)),
    batch_size=BATCH_SIZE, shuffle=False
)
train_eval_loader_bert = DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_train_eval_bert, dtype=torch.long), torch.tensor(y_train_eval_bert, dtype=torch.long)),
    batch_size=BATCH_SIZE, shuffle=False
)

In [ ]:
class BERT4Rec(nn.Module):
    def __init__(self, num_items, mask_token, max_len=MAX_LEN, d_model=64, n_heads=2, n_layers=2,
                 dropout=0.2, emb_dropout=0.5):
        super().__init__()
        vocab_size = num_items + 1  # +1 row reserved for [MASK] token id = num_items
        self.item_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.emb_dropout = nn.Dropout(emb_dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.layer_norm = nn.LayerNorm(d_model)
        self.max_len = max_len
        self.mask_token = mask_token
        self.num_items = num_items

    def forward(self, item_seq):
        batch_size, seq_len = item_seq.shape
        positions = torch.arange(seq_len, device=item_seq.device).unsqueeze(0).expand(batch_size, -1)

        x = self.item_emb(item_seq) + self.pos_emb(positions)
        x = self.emb_dropout(x)

        # NO causal mask here — this is the core difference from SASRec: every position
        # can attend to both earlier AND later positions (bidirectional context).
        padding_mask = (item_seq == 0)
        x = self.encoder(x, src_key_padding_mask=padding_mask)

        x = torch.nan_to_num(x, nan=0.0)
        x = self.layer_norm(x)
        return x

    def real_item_embeddings(self):
        """Embedding matrix restricted to real items (0=padding included, [MASK] row excluded) —
        use this whenever scoring against the real catalog, so [MASK] can never be recommended."""
        return self.item_emb.weight[:self.num_items]

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BERT4Rec(num_items=NUM_ITEMS, mask_token=MASK_TOKEN, max_len=MAX_LEN,
                  d_model=64, n_heads=2, n_layers=2, dropout=0.2, emb_dropout=0.5).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=1, factor=0.5)

print(f"Device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training

In [ ]:
import wandb

wandb.init(
    project="recsys-sequential",
    name="bert4rec-v1-dmodel64",
    config={
        "model": "BERT4Rec", "max_len": MAX_LEN, "d_model": 64, "n_heads": 2, "n_layers": 2,
        "dropout": 0.2, "emb_dropout": 0.5, "mask_prob": MASK_PROB, "batch_size": BATCH_SIZE,
        "lr": optimizer.param_groups[0]['lr'], "weight_decay": optimizer.param_groups[0]['weight_decay'],
    }
)

EPOCHS = 60
PATIENCE = 5
MIN_DELTA = 0.0
MIN_EPOCHS_BEFORE_STOPPING = 10
WARMUP_EPOCHS = 5
best_val_map = -float('inf')
patience_counter = 0
best_state = None

mask_rng = np.random.default_rng(42)  # separate RNG for masking, seeded but advances each epoch (dynamic masking)

for epoch in range(EPOCHS):
    # --- Rebuild masked training batch fresh every epoch: dynamic masking is what makes
    # BERT4Rec training effective — reusing one static mask pattern is much weaker ---
    X_train_bert, Y_train_bert = build_bert4rec_training_batch(train_sequences, max_len=MAX_LEN,
                                                                  mask_prob=MASK_PROB, rng=mask_rng)
    train_loader_bert = DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(X_train_bert, dtype=torch.long), torch.tensor(Y_train_bert, dtype=torch.long)),
        batch_size=BATCH_SIZE, shuffle=True
    )

    model.train()
    train_loss = 0.0
    total_grad_norm = 0.0

    for x_batch, y_batch in train_loader_bert:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        hidden = model(x_batch)

        hidden_flat = hidden.view(-1, hidden.size(-1))
        targets_flat = y_batch.view(-1)
        mask = targets_flat != 0          # only positions that were actually masked contribute to loss
        hidden_valid = hidden_flat[mask]
        targets_valid = targets_flat[mask]

        logits, labels = sampled_logits_and_labels(model, hidden_valid, targets_valid, NUM_ITEMS, num_negatives=100)
        loss = criterion(logits, labels)
        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        total_grad_norm += grad_norm.item()

        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader_bert)
    avg_grad_norm = total_grad_norm / len(train_loader_bert)

    # --- Validation: leave-one-out via appended [MASK], take hidden[:, -1, :] (the MASK position) ---
    model.eval()
    val_loss = 0.0
    all_hidden, all_targets = [], []

    with torch.no_grad():
        for x_batch, y_batch in val_loader_bert:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            hidden = model(x_batch)

            hidden_valid = hidden[:, -1, :]   # last position = [MASK] position, by construction
            targets_valid = y_batch

            all_hidden.append(hidden_valid.cpu())
            all_targets.append(targets_valid.cpu())

            logits, labels = sampled_logits_and_labels(model, hidden_valid, targets_valid, NUM_ITEMS, num_negatives=100)
            loss = criterion(logits, labels)
            val_loss += loss.item()

    val_loss /= len(val_loader_bert)

    all_hidden = torch.cat(all_hidden).to(device)
    all_targets = torch.cat(all_targets).to(device)
    val_map5 = sampled_ranking_map_at_k(model, all_hidden, all_targets, NUM_ITEMS, num_negatives=999, k=5, seed=42)

    all_hidden_train_eval, all_targets_train_eval = [], []
    with torch.no_grad():
        for x_batch, y_batch in train_eval_loader_bert:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            hidden = model(x_batch)
            all_hidden_train_eval.append(hidden[:, -1, :].cpu())
            all_targets_train_eval.append(y_batch.cpu())

    all_hidden_train_eval = torch.cat(all_hidden_train_eval).to(device)
    all_targets_train_eval = torch.cat(all_targets_train_eval).to(device)
    train_map5 = sampled_ranking_map_at_k(model, all_hidden_train_eval, all_targets_train_eval, NUM_ITEMS, num_negatives=999, k=5, seed=42)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    wandb.log({
        "epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss,
        "grad_norm": avg_grad_norm, "learning_rate": current_lr,
        "val_map@5": val_map5, "train_map@5": train_map5,
        "overfit_gap_map@5": train_map5 - val_map5,
    })

    print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Train MAP@5: {train_map5:.4f} | Val MAP@5: {val_map5:.4f} | "
          f"Gap: {train_map5 - val_map5:.4f} | Grad Norm: {avg_grad_norm:.4f}")

    if epoch + 1 > WARMUP_EPOCHS and val_map5 > best_val_map + MIN_DELTA:
        best_val_map = val_map5
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if epoch + 1 >= MIN_EPOCHS_BEFORE_STOPPING and patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(best_state)
wandb.finish()

## Validate on test set

In [ ]:
test_combined = pd.concat([train_df, valid_df, test_df])
test_sequences = build_user_sequences(test_combined)

X_test_bert, y_test = build_bert4rec_eval_batch(test_sequences, max_len=MAX_LEN)
print(f"Test: X={X_test_bert.shape}, y={len(y_test)}")

model.eval()
test_loader_bert = DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_test_bert, dtype=torch.long), torch.tensor(y_test, dtype=torch.long)),
    batch_size=BATCH_SIZE, shuffle=False
)

all_hidden_test, all_targets_test = [], []
with torch.no_grad():
    for x_batch, y_batch in test_loader_bert:
        x_batch = x_batch.to(device)
        hidden = model(x_batch)
        all_hidden_test.append(hidden[:, -1, :].cpu())   # [MASK] position
        all_targets_test.append(y_batch)

all_hidden_test = torch.cat(all_hidden_test).to(device)
all_targets_test = torch.cat(all_targets_test).to(device)
print(f"all_hidden_test: {all_hidden_test.shape}, all_targets_test: {all_targets_test.shape}")

In [ ]:
@torch.no_grad()
def full_ranking_eval_bert4rec(model, hidden_all, targets_all, num_items, k_list=(5, 10), chunk_size=2048):
    """Same as full_ranking_eval, but restricts candidate embeddings to real items only
    (model.item_emb.weight has num_items+1 rows; the last row is [MASK] and must never
    be a recommendation candidate)."""
    max_k = max(k_list)
    all_item_emb = model.real_item_embeddings()   # shape (num_items, d_model) — MASK row excluded
    recommended_items_by_user = {}
    relevant_items_by_user = {}
    global_idx = 0

    for start in range(0, hidden_all.size(0), chunk_size):
        end = start + chunk_size
        hidden_chunk = hidden_all[start:end]
        targets_chunk = targets_all[start:end]

        logits_chunk = hidden_chunk @ all_item_emb.T

        target_domain_id_chunk = domain_id[targets_chunk]
        wrong_domain_mask = domain_id.unsqueeze(0) != target_domain_id_chunk.unsqueeze(1)
        logits_chunk = logits_chunk.masked_fill(wrong_domain_mask, float('-inf'))

        top_k_items = logits_chunk.topk(max_k, dim=1).indices

        for i in range(hidden_chunk.size(0)):
            recommended_items_by_user[global_idx] = top_k_items[i].tolist()
            relevant_items_by_user[global_idx] = [targets_chunk[i].item()]
            global_idx += 1

    return recommended_items_by_user, relevant_items_by_user

recommended_model, relevant_items = full_ranking_eval_bert4rec(model, all_hidden_test, all_targets_test, NUM_ITEMS, chunk_size=128)

model_metrics = {
    "MAP@5": map_at_k(recommended_model, relevant_items, k=5),
    "NDCG@10": average_metric_over_users(ndcg_at_k, recommended_model, relevant_items, k=10),
    "Recall@10": average_metric_over_users(recall_at_k, recommended_model, relevant_items, k=10),
}
print("Model:", model_metrics)

In [ ]:
# Rebuild baselines specifically against BERT4Rec's y_test ordering, to guarantee alignment
rng = np.random.default_rng(42)
recommended_popular_bert = {}
recommended_random_bert = {}
for i, target_item in enumerate(y_test):
    is_books_target = item_domain[target_item] == 'books'
    recommended_popular_bert[i] = top_popular_books if is_books_target else top_popular_movies
    pool = books_item_idx_np if is_books_target else movies_item_idx_np
    recommended_random_bert[i] = rng.choice(pool, size=10, replace=False).tolist()

In [ ]:
# --- Domain-split results for BERT4Rec (same format as SASRec's domain breakdown) ---
# Reuses recommended_model / relevant_items produced by full_ranking_eval_bert4rec above,
# and the same recommended_popular / recommended_random baselines (domain-restricted,
# built once from y_test — valid for any model, since they don't depend on model predictions).

books_users, movies_users = split_by_domain(relevant_items, item_domain)
print(f"Books users: {len(books_users)}, Movies users: {len(movies_users)}")

results_by_domain_bert4rec = {}
for domain_name, user_ids in (("Books", books_users), ("Movies", movies_users)):
    model_d = compute_metrics_for_subset(recommended_model, relevant_items, user_ids)
    popular_d = compute_metrics_for_subset(recommended_popular_bert, relevant_items, user_ids)
    random_d = compute_metrics_for_subset(recommended_random_bert, relevant_items, user_ids)

    rows = []
    for metric in model_d:
        model_val = model_d[metric]
        popular_val = popular_d[metric]
        random_val = random_d[metric]
        lift_popular = model_val / popular_val if popular_val > 0 else float('inf')
        lift_random = model_val / random_val if random_val > 0 else float('inf')
        rows.append({
            "Metric": metric,
            "Model": model_val,
            "Top-Popular": popular_val,
            "Random": random_val,
            "Lift vs Popular": lift_popular,
            "Lift vs Random": lift_random,
        })

    df_domain = pd.DataFrame(rows)
    formatted = df_domain.copy()
    for col in ["Model", "Top-Popular", "Random"]:
        formatted[col] = formatted[col].map(lambda v: f"{v:.6f}")
    for col in ["Lift vs Popular", "Lift vs Random"]:
        formatted[col] = formatted[col].map(lambda v: "∞" if v == float('inf') else f"×{v:.2f}")

    print(f"\n=== {domain_name} (BERT4Rec) ===")
    print(formatted.to_string(index=False))
    results_by_domain_bert4rec[domain_name] = df_domain

## Training on single-domain

In [ ]:
def build_single_domain_bert4rec_training_batch(user_sequences, num_items, mask_token, max_len=MAX_LEN, mask_prob=MASK_PROB, rng=None):
    """Same as build_bert4rec_training_batch, but num_items/mask_token are
    explicit params instead of reading the global NUM_ITEMS/MASK_TOKEN --
    those belong to the joint vocab, not a single-domain one."""
    rng = rng or np.random.default_rng()
    X, Y = [], []
    for seq in user_sequences.values():
        if len(seq) < 2:
            continue
        seq = seq[-max_len:] if len(seq) > max_len else seq
        pad = max_len - len(seq)
        inp = [0] * pad + list(seq)
        tgt = [0] * max_len

        real_positions = list(range(pad, max_len))
        n_mask = max(1, int(round(len(real_positions) * mask_prob)))
        mask_positions = rng.choice(real_positions, size=min(n_mask, len(real_positions)), replace=False)

        for i in mask_positions:
            tgt[i] = inp[i]
            r = rng.random()
            if r < 0.8:
                inp[i] = mask_token
            elif r < 0.9:
                inp[i] = int(rng.integers(1, num_items))
        X.append(inp)
        Y.append(tgt)
    return np.array(X), np.array(Y)


@torch.no_grad()
def full_ranking_eval_bert4rec_single_domain(model, hidden_all, targets_all, num_items, k_list=(5, 10), chunk_size=2048):
    """Same as full_ranking_eval_bert4rec, minus the cross-domain masking --
    every item already belongs to the same domain here."""
    max_k = max(k_list)
    all_item_emb = model.real_item_embeddings()
    recommended_items_by_user, relevant_items_by_user = {}, {}
    global_idx = 0
    for start in range(0, hidden_all.size(0), chunk_size):
        end = start + chunk_size
        hidden_chunk = hidden_all[start:end]
        targets_chunk = targets_all[start:end]
        logits_chunk = hidden_chunk @ all_item_emb.T
        top_k_items = logits_chunk.topk(max_k, dim=1).indices
        for i in range(hidden_chunk.size(0)):
            recommended_items_by_user[global_idx] = top_k_items[i].tolist()
            relevant_items_by_user[global_idx] = [targets_chunk[i].item()]
            global_idx += 1
    return recommended_items_by_user, relevant_items_by_user


def train_bert4rec_single_domain(domain_name, raw_train_df, raw_valid_df, raw_test_df, epochs=60):
    print(f"\n{'='*20} BERT4Rec: {domain_name} {'='*20}")

    # Same MIN_ITEM_FREQ filter as the joint model (cell with frequent_items),
    # but computed within this domain alone -- the joint filter used cross-domain
    # combined counts, not valid for a single domain.
    train_df_d = raw_train_df.copy()
    valid_df_d = raw_valid_df.copy()
    test_df_d = raw_test_df.copy()
    counts = train_df_d['item_id'].value_counts()
    frequent = set(counts[counts >= MIN_ITEM_FREQ].index)
    train_df_d = train_df_d[train_df_d['item_id'].isin(frequent)].reset_index(drop=True)
    valid_df_d = valid_df_d[valid_df_d['item_id'].isin(frequent)].reset_index(drop=True)
    test_df_d = test_df_d[test_df_d['item_id'].isin(frequent)].reset_index(drop=True)

    domain_encoder = LabelEncoder()
    domain_encoder.fit(pd.concat([train_df_d['item_id'], valid_df_d['item_id'], test_df_d['item_id']]).unique())
    domain_num_items = len(domain_encoder.classes_) + 1
    domain_mask_token = domain_num_items
    for df in (train_df_d, valid_df_d, test_df_d):
        df['item_idx'] = domain_encoder.transform(df['item_id']) + 1

    train_sequences_d = build_user_sequences(train_df_d)
    valid_sequences_d = build_user_sequences(pd.concat([train_df_d, valid_df_d]))

    X_val, y_val = build_bert4rec_eval_batch(valid_sequences_d, max_len=MAX_LEN, mask_token=domain_mask_token)
    X_train_eval, y_train_eval = build_bert4rec_eval_batch(train_sequences_d, max_len=MAX_LEN, mask_token=domain_mask_token)
    val_loader_d = DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(X_val, dtype=torch.long), torch.tensor(y_val, dtype=torch.long)),
        batch_size=BATCH_SIZE, shuffle=False)
    train_eval_loader_d = DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(X_train_eval, dtype=torch.long), torch.tensor(y_train_eval, dtype=torch.long)),
        batch_size=BATCH_SIZE, shuffle=False)

    domain_model = BERT4Rec(num_items=domain_num_items, mask_token=domain_mask_token, max_len=MAX_LEN,
                             d_model=64, n_heads=2, n_layers=2, dropout=0.2, emb_dropout=0.5).to(device)
    domain_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    domain_optimizer = torch.optim.Adam(domain_model.parameters(), lr=1e-3, weight_decay=1e-4)
    domain_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(domain_optimizer, mode='min', patience=1, factor=0.5)

    wandb.init(
        project="recsys-sequential", name=f"bert4rec-{domain_name}-only-dmodel64",
        config={"model": "BERT4Rec", "domain": domain_name, "max_len": MAX_LEN, "d_model": 64,
                "n_heads": 2, "n_layers": 2, "dropout": 0.2, "emb_dropout": 0.5,
                "mask_prob": MASK_PROB, "batch_size": BATCH_SIZE},
    )

    PATIENCE, MIN_DELTA, MIN_EPOCHS_BEFORE_STOPPING, WARMUP_EPOCHS = 5, 0.0, 10, 5
    best_val_map, patience_counter, best_state = -float('inf'), 0, None
    mask_rng = np.random.default_rng(42)

    for epoch in range(epochs):
        X_train_bert, Y_train_bert = build_single_domain_bert4rec_training_batch(
            train_sequences_d, num_items=domain_num_items, mask_token=domain_mask_token,
            max_len=MAX_LEN, mask_prob=MASK_PROB, rng=mask_rng)
        train_loader_d = DataLoader(
            torch.utils.data.TensorDataset(torch.tensor(X_train_bert, dtype=torch.long), torch.tensor(Y_train_bert, dtype=torch.long)),
            batch_size=BATCH_SIZE, shuffle=True)

        domain_model.train()
        train_loss, total_grad_norm = 0.0, 0.0
        for x_batch, y_batch in train_loader_d:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            domain_optimizer.zero_grad()
            hidden = domain_model(x_batch)
            hidden_flat = hidden.view(-1, hidden.size(-1))
            targets_flat = y_batch.view(-1)
            mask = targets_flat != 0
            logits, labels = sampled_logits_and_labels(domain_model, hidden_flat[mask], targets_flat[mask], domain_num_items, num_negatives=100)
            loss = domain_criterion(logits, labels)
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(domain_model.parameters(), max_norm=2.0)
            total_grad_norm += grad_norm.item()
            domain_optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_d)
        avg_grad_norm = total_grad_norm / len(train_loader_d)

        domain_model.eval()
        val_loss = 0.0
        all_hidden, all_targets = [], []
        with torch.no_grad():
            for x_batch, y_batch in val_loader_d:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                hidden = domain_model(x_batch)
                hidden_valid, targets_valid = hidden[:, -1, :], y_batch
                all_hidden.append(hidden_valid.cpu())
                all_targets.append(targets_valid.cpu())
                logits, labels = sampled_logits_and_labels(domain_model, hidden_valid, targets_valid, domain_num_items, num_negatives=100)
                val_loss += domain_criterion(logits, labels).item()
        val_loss /= len(val_loader_d)
        all_hidden, all_targets = torch.cat(all_hidden).to(device), torch.cat(all_targets).to(device)
        val_map5 = sampled_ranking_map_at_k(domain_model, all_hidden, all_targets, domain_num_items, num_negatives=999, k=5, seed=42)

        all_hidden_te, all_targets_te = [], []
        with torch.no_grad():
            for x_batch, y_batch in train_eval_loader_d:
                x_batch = x_batch.to(device)
                hidden = domain_model(x_batch)
                all_hidden_te.append(hidden[:, -1, :].cpu())
                all_targets_te.append(y_batch)
        all_hidden_te, all_targets_te = torch.cat(all_hidden_te).to(device), torch.cat(all_targets_te).to(device)
        train_map5 = sampled_ranking_map_at_k(domain_model, all_hidden_te, all_targets_te, domain_num_items, num_negatives=999, k=5, seed=42)

        domain_scheduler.step(val_loss)
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss,
                    "grad_norm": avg_grad_norm, "learning_rate": domain_optimizer.param_groups[0]['lr'],
                    "val_map@5": val_map5, "train_map@5": train_map5, "overfit_gap_map@5": train_map5 - val_map5})
        print(f"[{domain_name}] Epoch [{epoch+1}/{epochs}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Train MAP@5: {train_map5:.4f} | Val MAP@5: {val_map5:.4f} | Gap: {train_map5 - val_map5:.4f}")

        if epoch + 1 > WARMUP_EPOCHS and val_map5 > best_val_map + MIN_DELTA:
            best_val_map = val_map5
            best_state = {k: v.cpu().clone() for k, v in domain_model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if epoch + 1 >= MIN_EPOCHS_BEFORE_STOPPING and patience_counter >= PATIENCE:
                print(f"[{domain_name}] Early stopping at epoch {epoch+1}")
                break

    domain_model.load_state_dict(best_state)
    wandb.finish()

    test_sequences_d = build_user_sequences(pd.concat([train_df_d, valid_df_d, test_df_d]))
    X_test, y_test_d = build_bert4rec_eval_batch(test_sequences_d, max_len=MAX_LEN, mask_token=domain_mask_token)
    domain_model.eval()
    test_loader_d = DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(X_test, dtype=torch.long), torch.tensor(y_test_d, dtype=torch.long)),
        batch_size=BATCH_SIZE, shuffle=False)
    all_hidden_test, all_targets_test = [], []
    with torch.no_grad():
        for x_batch, y_batch in test_loader_d:
            hidden = domain_model(x_batch.to(device))
            all_hidden_test.append(hidden[:, -1, :].cpu())
            all_targets_test.append(y_batch)
    all_hidden_test, all_targets_test = torch.cat(all_hidden_test).to(device), torch.cat(all_targets_test).to(device)

    recommended_d, relevant_d = full_ranking_eval_bert4rec_single_domain(domain_model, all_hidden_test, all_targets_test, domain_num_items, chunk_size=128)
    model_metrics_d = {
        "MAP@5": map_at_k(recommended_d, relevant_d, k=5),
        "NDCG@10": average_metric_over_users(ndcg_at_k, recommended_d, relevant_d, k=10),
        "Recall@10": average_metric_over_users(recall_at_k, recommended_d, relevant_d, k=10),
    }

    real_items = np.arange(1, domain_num_items)
    top_popular_d = train_df_d['item_idx'].value_counts().index[:10].tolist()
    rng_bl = np.random.default_rng(42)
    recommended_popular_d = {i: top_popular_d for i in range(len(y_test_d))}
    recommended_random_d = {i: rng_bl.choice(real_items, size=10, replace=False).tolist() for i in range(len(y_test_d))}
    popular_metrics_d = {
        "MAP@5": map_at_k(recommended_popular_d, relevant_d, k=5),
        "NDCG@10": average_metric_over_users(ndcg_at_k, recommended_popular_d, relevant_d, k=10),
        "Recall@10": average_metric_over_users(recall_at_k, recommended_popular_d, relevant_d, k=10),
    }
    random_metrics_d = {
        "MAP@5": map_at_k(recommended_random_d, relevant_d, k=5),
        "NDCG@10": average_metric_over_users(ndcg_at_k, recommended_random_d, relevant_d, k=10),
        "Recall@10": average_metric_over_users(recall_at_k, recommended_random_d, relevant_d, k=10),
    }
    print(f"[{domain_name}] Model: {model_metrics_d} | Top-Popular: {popular_metrics_d} | Random: {random_metrics_d}")

    return model_metrics_d, popular_metrics_d, random_metrics_d

In [ ]:
books_only_metrics, books_only_popular_metrics, books_only_random_metrics = train_bert4rec_single_domain(
    'books', books_train, books_valid, books_test, epochs=60,
)
movies_only_metrics, movies_only_popular_metrics, movies_only_random_metrics = train_bert4rec_single_domain(
    'movies', movies_train, movies_valid, movies_test, epochs=60,
)

## Saving metrics results

In [ ]:
def bert4rec_result_row_from_metrics(model_name, metrics_dict):
    row = {
        'model': model_name, 'd_model': 64, 'n_heads': 2, 'n_layers': 2, 'emb_dropout': 0.5,
        'k': 10, 'map_k': 5, 'n_users': train_df['user_id'].nunique(), 'n_items': NUM_ITEMS - 1,
    }
    for metric_label, key in [('MAP@5', 'map@5'), ('NDCG@10', 'ndcg@10'), ('Recall@10', 'recall@10')]:
        row[f'test_{key}'] = round(metrics_dict[metric_label], 4)
    return row

bert4rec_metrics_results = pd.DataFrame([
    bert4rec_result_row('bert4rec_books_movies_joint', 'Model'),
    bert4rec_result_row_from_metrics('bert4rec_books_only', books_only_metrics),
    bert4rec_result_row_from_metrics('bert4rec_movies_only', movies_only_metrics),
    bert4rec_result_row('popularity_baseline', 'Top-Popular'),
    bert4rec_result_row('random_baseline', 'Random'),
])

RESULTS_DIR = f'{DATA_DIR}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
bert4rec_metrics_results.to_csv(f'{RESULTS_DIR}/bert4rec_metrics.csv', index=False)
bert4rec_metrics_results

## Export artifacts for the app MVP


In [ ]:
import json

ARTIFACTS_DIR = f'{DATA_DIR}/artifacts/bert4rec'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

torch.save(model.state_dict(), f'{ARTIFACTS_DIR}/model_state_dict.pt')

# Same vocabulary as SASRec (both trained on the same encoder/train_df) --
# duplicated here so each artifact directory is self-contained.
item2idx = {cls: i + 1 for i, cls in enumerate(encoder.classes_)}
with open(f'{ARTIFACTS_DIR}/item2idx.json', 'w') as f:
    json.dump(item2idx, f)

with open(f'{ARTIFACTS_DIR}/domain_item_indices.json', 'w') as f:
    json.dump({
        'books': books_item_idx.cpu().tolist(),
        'movies': movies_item_idx.cpu().tolist(),
    }, f)

with open(f'{ARTIFACTS_DIR}/config.json', 'w') as f:
    json.dump({
        'num_items': NUM_ITEMS,
        'max_len': MAX_LEN,
        'mask_token': MASK_TOKEN,
        'd_model': 64,
        'n_heads': 2,
        'n_layers': 2,
        'dropout': 0.2,
        'emb_dropout': 0.5,
    }, f)

print(f'Saved BERT4Rec artifacts to {ARTIFACTS_DIR}')
